[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/hopf/hopf.ipynb)

# The Hopf Fibration

This example computes and draws the Hopf fibration (Hopf, 1931), which groups the unit spinors of three-dimensional space by their spin on the unit sphere. Each unit spinor turns the north pole of the sphere into another point on the sphere, called its spin, while a preliminary turn in the equatorial plane changes the spinor without moving that pole. The unit spinors sharing one spin therefore form a circle on the three-sphere, the fibre over that spin, and stereographic projection into three-dimensional space sends those fibres to nested tori of circles in which every pair of fibres links once. Carrying a spinor around a closed loop of spins by minimal-angle rotations returns the spinor to its starting fibre shifted along that circle by half the solid angle enclosed by the loop.

Spinors are modelled as unit even multivectors and spins as intrinsic points on the unit sphere, so sandwiching the north `pole` with the spinor space writes the quadratic Hopf map as `hopf = Even >> pole`. Pairing that form with a target spin via `spin.dot(hopf)` yields a symmetric scalar form whose top eigenspace spans the fibre over that spin, and right-multiplying a spinor by `(xy * angle).exp()` sweeps around the fibre because turns in `xy` leave `pole` fixed. Stereographic projection maps each spinor's bivector grade to a point in space through an intrinsic scalar `handedness`, where the Gauss linking number reduces to summing the regressive volume `separation & step_first & step_second` over segment pairs.

In complex-coordinate and Pauli-matrix notation, a unit spinor is parameterized by $(z_1, z_2) \in \mathbb{C}^2$ with $|z_1|^2 + |z_2|^2 = 1$, mapped to the unit sphere in $\mathbb{R}^3$ component by component as $(2\operatorname{Re}(z_1\bar z_2),\, 2\operatorname{Im}(z_1\bar z_2),\, |z_1|^2 - |z_2|^2) = \langle\psi|\boldsymbol{\sigma}|\psi\rangle$ and inverted onto a fibre by choosing a gauge patch away from the south-pole singularity $z_2 = 0$ with phase factor $e^{i\chi}$. In Gibbs–Heaviside vector notation, stereographic projection identifies the three imaginary quaternion components $(q_x, q_y, q_z) / (1 + q_0)$ directly with Cartesian vectors, and the Gauss linking integral evaluates the scalar triple product $\frac{1}{4\pi}\oint\oint \frac{(\mathbf{r}_1 - \mathbf{r}_2)\cdot(d\mathbf{r}_1 \times d\mathbf{r}_2)}{|\mathbf{r}_1 - \mathbf{r}_2|^3}$.

In [ ]:
# Install numga and its examples when running in Colab.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import numpy as np

from examples.math.hopf import render
from examples.util.animation import save_figure, show_states
from numga import NumpyContext
from numga.algebras import VGA3D

np.set_printoptions(precision=4, suppress=True)

# Geometric algebra of three-dimensional Euclidean space.
ga = VGA3D
mv = NumpyContext(ga).multivector
Antivector = ga.gatype.antivector()                  # Antivector-
HandedScalar = ga.gatype.antipseudoscalar()          # Scalar-
Even = ga.gatype.even()                              # Even+
Rotor = ga.gatype.rotor()                            # Rotor+

## A spinor's spin as a quadratic form

Sandwiching the north `pole` (`z.dual()`) with a unit spinor rotates that pole into the spinor's spin on the unit sphere. Applying the same sandwich to the spinor type itself produces the quadratic map `hopf = Even >> pole`, an antivector-valued bilinear form whose value on `(psi, psi)` is the spin of `psi`. Right-multiplying `psi` by a rotor in `xy` rotates `pole` in its own plane before `psi` acts, leaving the resulting spin unchanged while advancing `psi` along its fibre.

In [ ]:
psi = mv.even([0.4, -0.3, 0.7, 0.5]).normalized()                               # [] Even+
turn_angle = 1.1

In [ ]:
# North pole of the unit spin sphere and quadratic Hopf map on pairs of spinors.
pole = mv.z.dual()                                                              # [] Antivector-
hopf = Even >> pole                                                             # [] Antivector- <- (Even+, Even+)

spin = hopf(psi, psi)                                                           # [] Antivector-
right_turn = (mv.xy * turn_angle).exp()                                         # [] Rotor+
turned_psi = psi * right_turn                                                   # [] Even+

In [ ]:
print(hopf)
print("spin:\n", spin.coefficient_table())
print("turned on the right:\n", hopf(turned_psi, turned_psi).coefficient_table())
print("psi >> pole:\n", (psi >> pole).coefficient_table())

## The fibre over a spin

Taking the inner product `spin.dot(hopf)` projects the quadratic Hopf map onto a target `spin`, yielding a symmetric scalar form on spinors that measures alignment with `spin`. Its eigenvalues are `-1, -1, 1, 1`, so its two-dimensional `+1` eigenspace intersects the unit three-sphere in the circular fibre of spinors over `spin`. Right-multiplying one unit spinor from that eigenspace by `(xy * angle).exp()` traces the full circle of the fibre.

In [ ]:
n_circle_points = 6

In [ ]:
def fibre_start(spin: Antivector) -> Even:
    """The top eigenvector of `spin.dot(hopf)` selects a unit spinor (`[...] Even+`) on the fibre over each `spin`."""
    _, spinors = spin.dot(hopf).eigh()                                          # [..., 4] Even+
    return spinors[..., -1]                                                     # [...] Even+


values, _ = spin.dot(hopf).eigh()                                               # [4] Scalar+
start = fibre_start(spin)                                                       # [] Even+
circle_turns = (mv.xy * np.linspace(0.0, 2 * np.pi, n_circle_points + 1)).exp() # [n_circle_points + 1] Rotor+
circle = start * circle_turns                                                   # [n_circle_points + 1] Even+

In [ ]:
print("eigenvalues of spin.dot(hopf):", values.to_array())

## Nested tori under stereographic projection

Stereographic projection from the spinor `-1` maps the unit three-sphere into three-dimensional space by dividing each spinor's bivector grade by `1 + spinor.select[0]` and multiplying by an intrinsic scalar `handedness` so the projected bivector becomes an intrinsic spatial point. Great circles of the three-sphere project to circles in space, and the fibres over each latitude circle of spins sweep out a torus, with different latitudes nesting inside one another.

In [ ]:
torus_polars = np.pi * np.array([0.9, 0.7, 0.5])
n_per_circle = 18
n_samples = 240

In [ ]:
# Intrinsic unit scalar fixing the right-hand rule from bivector planes to spatial points.
handedness = mv(HandedScalar, [1])                                              # [] Scalar-


def sphere_spins(polar: np.ndarray, azimuth: np.ndarray) -> Antivector:
    """Polar tilts in `zx` followed by azimuthal turns in `xy` carry `pole` to unit spins (`[...] Antivector-`) on the sphere."""
    tilts = (mv.zx * (-polar / 2)).exp()                                        # [...] Rotor+
    turns = (mv.xy * (-azimuth / 2)).exp()                                      # [...] Rotor+
    return (turns * tilts) >> pole                                              # [...] Antivector-


def stereographic_projection(spinor: Even) -> Antivector:
    """Stereographic projection from `-1` maps unit spinors into three-dimensional spatial points (`[...] Antivector-`)."""
    # Scale the bivector grade by 1 / (1 + scalar) and convert to an intrinsic point via handedness.
    return handedness * spinor.select[2] / (1 + spinor.select[0])               # [...] Antivector-


azimuths = np.linspace(0.0, 2 * np.pi, n_per_circle, endpoint=False)
spins = sphere_spins(torus_polars[:, None], azimuths[None, :])                  # [3, n_per_circle] Antivector-
fibre_turns = (mv.xy * np.linspace(0.0, 2 * np.pi, n_samples + 1)).exp()        # [n_samples + 1] Rotor+
fibres = fibre_start(spins)[..., None] * fibre_turns                            # [3, n_per_circle, n_samples + 1] Even+
projected_fibres = stereographic_projection(fibres)                             # [3, n_per_circle, n_samples + 1] Antivector-
save_figure(render.tori_figure(spins, projected_fibres), "hopf_tori");

The inset sphere on the left marks the three latitude circles of spins coloured by azimuth and shaded toward the south pole, while the right panel draws each spin's projected fibre in the matching colour to reveal the three nested Hopf tori.

## Gauss linking number of projected fibres

Two closed curves in three-dimensional space are linked when neither can be pulled free without crossing the other, and their linking number counts the signed crossings. Discretizing two fibres as closed polygons, the Gauss linking sum pairs every segment of the first curve with every segment of the second through the three-way regressive volume `separation & step_first & step_second` divided by the cubed separation distance, evaluating to `+1` against `handedness` for every pair of distinct Hopf fibres.

In vector-calculus notation, this sum discretizes Gauss's double line integral $\frac{1}{4\pi}\oint\oint \frac{(\mathbf{r}_1 - \mathbf{r}_2)\cdot(d\mathbf{r}_1 \times d\mathbf{r}_2)}{|\mathbf{r}_1 - \mathbf{r}_2|^3}$, where the cross product and dot product fix a handedness implicitly.

In [ ]:
first_circles, first_places = np.array([0, 0, 0, 1]), np.array([0, 0, 4, 7])
second_circles, second_places = np.array([0, 1, 2, 2]), np.array([5, 3, 11, 2])

In [ ]:
def linking_number(first: Antivector, second: Antivector) -> HandedScalar:
    """The discrete Gauss sum over segment pairs gives the signed linking number (`[...] Scalar-`) between closed polygonal curves."""
    step_first = first[..., 1:] - first[..., :-1]                               # [...] Antivector-
    step_second = second[..., 1:] - second[..., :-1]                            # [...] Antivector-
    middle_first = 0.5 * (first[..., 1:] + first[..., :-1])                     # [...] Antivector-
    middle_second = 0.5 * (second[..., 1:] + second[..., :-1])                  # [...] Antivector-
    separation = middle_first[..., :, None] - middle_second[..., None, :]       # [...] Antivector-
    # Regressive product of separation and both segment displacements spans their signed volume.
    volume = separation & step_first[..., :, None] & step_second[..., None, :]  # [...] Scalar-
    distance = separation.antinorm()                                            # [...] Scalar+
    return (volume / distance ** 3).sum(axis=(-2, -1)) / (4 * np.pi)            # [...] Scalar-


linking_numbers = linking_number(
    projected_fibres[first_circles, first_places],
    projected_fibres[second_circles, second_places],
)                                                                               # [4] Scalar-

In [ ]:
print("linking numbers:", linking_numbers.to_array())

The sign of this linking number rests on more choices than the side of the turns. The conventional construction identifies the bivector part of a spinor with a spatial position directly along `x`, `y`, and `z`; its fibres link `+1`, the fibres of `psi.reverse() >> pole` with left turns link `-1`, and the mirror image of the link in space also links `-1`. Here the projection reaches an intrinsic spatial point through `handedness`, so the linking number of points is an intrinsic scalar equal to `+1` for right turns, `-1` for left turns, and `-1` for the mirrored points `x >> projected_fibres`, all measured against `handedness`. Mirroring the spinors first and projecting with the same `handedness`, `stereographic_projection(x >> fibres)`, gives `+1` because `handedness` is a fixed constant of the projection that mirroring the spinors does not reverse.

Left as extrinsic bivectors, the projected fibres would require no `handedness`. Three bivectors span an extrinsic volume, a spatial reflection acts on bivectors as a rotation, and their linking number would remain `+1` under `x >> projected_fibres`, determined by the side of the turns alone. Typing spatial positions as intrinsic introduces a handedness choice when identifying turning planes with points in space, reflecting the chirality of the projected Hopf link against its spatial mirror image.

## Fibres swept along a spiral of spins

Following a single spin as it spirals upward from near the south pole toward the equator adds one projected circle per step, showing how successive fibres interlock to fill out the nested tori.

In [ ]:
spiral_polars = np.pi * np.array([0.95, 0.45])
n_turns = 4
n_frames = 68
frame_duration_ms = 40

In [ ]:
def fibre_sweep(polars: np.ndarray, n_turns: int, n_frames: int,
                n_samples: int) -> Iterator[tuple[Antivector, Antivector]]:
    """Yields `(spin, projected_fibre)` pairs as `spin` spirals between `polars[0]` and `polars[1]`."""
    fibre_turns = (mv.xy * np.linspace(0.0, 2 * np.pi, n_samples + 1)).exp()    # [n_samples + 1] Rotor+
    for fraction in np.linspace(0.0, 1.0, n_frames):
        polar = polars[0] + (polars[1] - polars[0]) * fraction
        spiral_spin = sphere_spins(polar, 2 * np.pi * n_turns * fraction)       # [] Antivector-
        fibre = fibre_start(spiral_spin) * fibre_turns                          # [n_samples + 1] Even+
        yield spiral_spin, stereographic_projection(fibre)


sweep_states = show_states(
    fibre_sweep(spiral_polars, n_turns, n_frames, n_samples),
    render.sweep_animation,
    name="hopf_sweep",
    duration_ms=frame_duration_ms,
);

Each newly added fibre appears in a thicker stroke coloured by its spin dot on the inset sphere, threading once through every earlier circle as the spiral climbs.

## Parallel transport around a loop of spins

Carrying a spinor around a closed latitude loop of spins by applying the minimal-angle rotor between consecutive spins transports the spinor horizontally across the fibration. Because the loop closes on the sphere, the final spinor returns to the fibre over its starting spin shifted along that circle by half the solid angle enclosed by the loop.

In bra-ket notation, this holonomy shift is the Berry phase $\gamma = \Omega / 2$ of a spin-$\tfrac{1}{2}$ state $|\psi\rangle$ transported adiabatically around a loop enclosing solid angle $\Omega$ under the Berry connection $i\langle\psi|d\psi\rangle$.

In [ ]:
loop_polar = 2.2
n_steps = 400

In [ ]:
def transport_rotors(spins: Antivector) -> Rotor:
    """Minimal-angle step rotors multiplied along `spins` carry a frame (`[...] Rotor+`) from `spins[0]` to each subsequent spin."""
    # Adding 1 halves the double-angle rotor spins[1:] * ~spins[:-1] between consecutive unit spins.
    step_rotors = (1 + spins[1:] * ~spins[:-1]).normalized()                    # [...] Rotor+
    return step_rotors.cumprod(axis=0)                                          # [...] Rotor+


loop_angles = np.linspace(0.0, 2 * np.pi, n_steps + 1)
loop = sphere_spins(loop_polar, loop_angles)                                    # [n_steps + 1] Antivector-
loop_start = fibre_start(loop[0])                                               # [] Even+
carried_spinors = transport_rotors(loop) * loop_start                           # [n_steps] Even+
holonomy_turn = loop_start.reverse() * carried_spinors[-1]                      # [] Even+
half_solid_angle = np.pi * (1 - np.cos(loop_polar))
start_fibre = loop_start * fibre_turns                                          # [n_samples + 1] Even+
save_figure(
    render.lift_figure(loop, stereographic_projection(carried_spinors), stereographic_projection(start_fibre)),
    "hopf_lift",
);

The red latitude circle on the inset sphere marks the closed path of spins, while the red curve in space leaves the blue starting marker on the grey initial fibre and returns to the red endpoint shifted partway around that same circle.

In [ ]:
expected_holonomy = (mv.xy * (-half_solid_angle)).exp()                         # [] Rotor+
print("turn along the fibre:\n", holonomy_turn.coefficient_table())
print("expected half-solid-angle rotor:\n", expected_holonomy.coefficient_table())

In [ ]:
# checks
assert ((spin - (psi >> pole)).antinorm() < 1e-12).all()
assert ((hopf(turned_psi, turned_psi) - spin).antinorm() < 1e-9).all()
assert ((values - np.array([-1, -1, 1, 1])).abs() < 1e-12).all()
assert ((hopf(circle, circle) - spin).antinorm() < 1e-9).all()
assert ((hopf(fibres, fibres) - spins[..., None]).antinorm() < 1e-9).all()
assert ((linking_numbers - handedness).abs() < 1e-2).all()
assert len(sweep_states) == n_frames
assert ((sweep_states[-1][0].antinorm() - 1).abs() < 1e-9).all()
assert ((holonomy_turn - expected_holonomy).norm() < 1e-4).all()